# Week 12 Lab — From differences to trends

**HWRS 564a · Fall 2026**

This compact statistics lab applies one workflow twice: estimate an effect, test it against zero, check assumptions, and limit the claim to the design.

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked **`# YOUR TURN`** need your code. Cells marked **`# CHECK`** verify the result. Before finishing, restart the kernel and run all cells in order.

## By the end you can

1. identify the independent observational unit;
2. calculate and report a paired effect in feet;
3. construct and interpret a sign-flip null distribution;
4. fit and diagnose a simple linear groundwater trend; and
5. write conclusions that state evidence without claiming a cause.

---

## The groundwater question

Tucson's water-supply operations shifted around 2000–01 as pumping moved toward Avra Valley recharge-and-recovery facilities and deliveries began to include recharged Colorado River water. The event date comes from operations history, not from searching the water-level record.

**Question:** Among Tucson-area wells monitored in both windows, how did median depth to water change from **1996–2000** to **2002–2006**?

The year 2001 is excluded as a transition. A well is included only when it has measurements in at least three distinct years of each window. Each well contributes one pre-period median and one post-period median.

> **Sign convention:** `post - pre`. Positive change means the water table was deeper below land surface in the later period.

Sources: [Arizona Department of Water Resources](https://www.azwater.gov/sites/default/files/2022-08/download.pdf) and [Tucson Water](https://www.tucsonaz.gov/Departments/Water/Water-Quality/CCR_MainSystem_2024-File).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path("../..")  # course repo root
DATA = ROOT / "data"

wells = pd.read_csv(
    DATA / "week12_tucson_paired_wells.csv",
    dtype={"site_no": str},
)
print(f"{len(wells)} rows and {wells['site_no'].nunique()} unique wells")
wells.head()

## The unit is a well, not a tape reading

The source data contain 866 readings, but measurements from one well share location, construction, screened interval, and aquifer history. Treating every row as independent would exaggerate the sample size.

Here the **observational unit is one monitored well**, so the sample size is 18 paired wells.

---

## Part 1 — Calculate the observed effect

For each well, calculate

$$d_i = \text{post median}_i - \text{pre median}_i.$$

The file includes this change, but recalculate it so the sign convention remains visible. Then find the mean change across the 18 wells.

### YOUR TURN 1 — Paired change and observed mean

Create `change_check_ft` from the two period medians and calculate `mean_change_ft`.

In [ ]:
# YOUR TURN
wells["change_check_ft"] = ...
mean_change_ft = ...

In [ ]:
# CHECK
assert wells["site_no"].is_unique, "each row should be one well"
assert np.allclose(wells["change_check_ft"], wells["change_depth_ft"])
assert np.isclose(mean_change_ft, 12.4733333333)
print(f"n = {len(wells)} wells")
print(f"mean paired change: {mean_change_ft:+.2f} ft")
print("Correct. Positive means deeper in the later period.")

The observed effect is **+12.47 ft**: on average, the later period median was about 12.5 ft deeper. Report that magnitude and unit before calculating a p-value.

---

## Part 2 — Show every well

A mean hides variation. Plot all 18 paired changes so a reader can see direction, spread, exceptions, and whether one well dominates the result.

### YOUR TURN 2 — Plot the paired changes

Make a horizontal dot plot with one point per well, a dashed line at zero, and a solid line at the observed mean.

In [ ]:
# YOUR TURN
fig, ax = ...
ax.scatter(..., ..., color="#1E5288")
ax.axvline(..., color="0.35", ls="--", label="no change")
ax.axvline(..., color="#AB0520", lw=3, label="observed mean")
ax.set_xlabel("post minus pre median depth (ft)")
ax.set_ylabel("well")
ax.set_yticks([])
ax.legend()
plt.show()

In [ ]:
# CHECK
assert len(ax.collections[0].get_offsets()) == len(wells), "show one point per well"
line_x = [line.get_xdata()[0] for line in ax.lines]
assert any(np.isclose(x, 0) for x in line_x), "include a zero reference line"
assert any(np.isclose(x, mean_change_ft) for x in line_x), "show the observed mean"
print("Correct. The observations remain visible behind the summary.")

---

## Part 3 — Build a no-effect distribution

The null hypothesis says there is **no systematic pre/post change**. Under that claim, pre and post labels can be swapped within a well. Swapping labels turns a paired change $d_i$ into $-d_i$.

For each repetition:

1. randomly keep or reverse each well's sign;
2. calculate the mean signed change; and
3. store it.

This keeps every well pair intact.

### YOUR TURN 3 — Permutation distribution and p-value

Generate 10,000 sign-flipped mean changes. The two-sided p-value is the fraction at least as far from zero as the observed mean. The plus-one correction prevents a finite simulation from reporting exactly zero.

In [ ]:
change = wells["change_depth_ft"].to_numpy()
observed = change.mean()
rng = np.random.default_rng(5647)
n_perm = 10_000

# YOUR TURN
null_means = ...
for i in range(n_perm):
    signs = ...
    null_means[i] = ...
p_perm = ...

In [ ]:
# CHECK
assert null_means.shape == (n_perm,)
assert abs(null_means.mean()) < 0.25, "the null distribution should center near zero"
assert 0 < p_perm < 0.005
print(f"observed mean = {observed:+.2f} ft")
print(f"two-sided permutation p = {p_perm:.4f}")
print("Correct.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(null_means, bins=35, color="#1E5288", edgecolor="white")
ax.axvline(observed, color="#AB0520", lw=3, label=f"observed = {observed:.2f} ft")
ax.axvline(-observed, color="#AB0520", lw=3)
ax.set_xlabel("mean change under no systematic effect (ft)")
ax.set_ylabel("shuffles")
ax.legend()
plt.show()

A p-value near 0.001 means: **if there were no systematic period effect and the signs were exchangeable, about 1 in 1,000 random sign assignments would produce a mean at least this far from zero.**

It is not the probability that the null is true, the size of the effect, or evidence of a causal mechanism.

---

## Part 4 — Fit one simple groundwater trend

A regression asks the same inferential question with a different effect: is the slope distinguishable from zero? Use one long-record well and express time in years so the slope has useful units. Positive depth means a declining water table.

### YOUR TURN 4 — Fit the line and calculate residuals

Use `stats.linregress` on elapsed years and depth to water. Then calculate the fitted values and residuals.

In [ ]:
levels = pd.read_csv(
    DATA / "tucson_water_levels.csv",
    dtype={"site_no": str},
    parse_dates=["date"],
)
one = (
    levels.loc[levels["site_no"] == "320824110593001"]
    .sort_values("date")
    .dropna(subset=["date", "depth_to_water_ft"])
    .copy()
)
one["years_elapsed"] = (one["date"] - one["date"].min()).dt.days / 365.25

# YOUR TURN
fit = ...
one["predicted"] = ...
one["residual"] = ...

In [ ]:
# CHECK
np.testing.assert_allclose(fit.slope, 2.0811353573)
np.testing.assert_allclose(one["residual"].mean(), 0.0, atol=1e-10)
print(f"slope: {fit.slope:+.2f} ft/year; p = {fit.pvalue:.4g}")
print("Correct.")

The fitted slope is descriptive until its assumptions are checked. A small p-value does not repair curvature, serial dependence, changing variance, an influential observation, or a nonrepresentative monitoring record.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(one["date"], one["depth_to_water_ft"], s=12, alpha=0.7)
axes[0].plot(one["date"], one["predicted"], color="#AB0520", lw=2)
axes[0].invert_yaxis()
axes[0].set(xlabel="date", ylabel="depth to water (ft)", title="Observed and fitted")
axes[1].scatter(one["date"], one["residual"], s=12, alpha=0.7)
axes[1].axhline(0, color="#AB0520", ls="--")
axes[1].set(xlabel="date", ylabel="residual (ft)", title="Residuals retain time structure")
fig.tight_layout()
plt.show()

---

## Write the bounded conclusion

Add a new Markdown cell below this one. Write one short conclusion for the paired comparison and one for the trend. In each:

1. state the effect, direction, and units before the p-value;
2. name the observational unit or time structure; and
3. give one design- or residual-based limitation.

Do **not** write that either small p-value proves a cause or guarantees a future trend. The paired comparison lacks an untreated control, and the regression residuals show that one straight line does not capture all decade-scale structure.

## Before you leave

1. Restart the kernel and run all completed cells in order.
2. Confirm that positive always means deeper.
3. Confirm that the sample size is 18 **wells**, not 866 readings.
4. Confirm the regression slope is in ft/year, not ft/day.
5. Read both conclusions without looking at the code.

## Next week

You will turn this evidence into one figure worth showing, with all observations visible and uncertainty communicated honestly.